# Дообучение

В этом руководстве показано, как дообучить предобученный checkpoint `GR00T-N1` на постобучающем датасете того же воплощения. Это демонстрирует пользу постобучения: универсальная модель становится экспертной и показывает прирост качества.

В этом руководстве используется демонстрационный датасет `robot_sim.PickNPlace` из папки [demo_data](./demo_data).

Сначала мы загрузим предобученную модель и оценим её на датасете. Затем дообучим модель на этом датасете и снова оценим качество.

## Предобученная модель

In [ ]:
from gr00t.utils.eval import calc_mse_for_single_trajectory
import warnings
from gr00t.experiment.data_config import DATA_CONFIG_MAP
from gr00t.model.policy import Gr00tPolicy
from gr00t.data.schema import EmbodimentTag
from gr00t.data.dataset import LeRobotSingleDataset
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

warnings.simplefilter("ignore", category=FutureWarning)

In [ ]:
PRE_TRAINED_MODEL_PATH = "nvidia/GR00T-N1-2B"
EMBODIMENT_TAG = EmbodimentTag.GR1
DATASET_PATH = "../demo_data/robot_sim.PickNPlace"


data_config = DATA_CONFIG_MAP["gr1_arms_only"]
modality_config = data_config.modality_config()
modality_transform = data_config.transform()


pre_trained_policy = Gr00tPolicy(
    model_path=PRE_TRAINED_MODEL_PATH,
    embodiment_tag=EMBODIMENT_TAG,
    modality_config=modality_config,
    modality_transform=modality_transform,
    device=device,
)

dataset = LeRobotSingleDataset(
    dataset_path=DATASET_PATH,
    modality_configs=modality_config,
    video_backend="decord",
    video_backend_kwargs=None,
    transforms=None,  # We'll handle transforms separately through the policy
    embodiment_tag=EMBODIMENT_TAG,
)


mse = calc_mse_for_single_trajectory(
    pre_trained_policy,
    dataset,
    traj_id=0,
    modality_keys=["right_arm", "right_hand"],   # we will only evaluate the right arm and right hand
    steps=150,
    action_horizon=16,
    plot=True
)

print("MSE loss for trajectory 0:", mse)

Отлично! Видны предсказанные и реальные действия. Предсказания не идеальны, но близки к реальным. Это показывает, что предобученный checkpoint работает хорошо.

Теперь возьмём 10 случайных траекторий и вычислим средний MSE, чтобы получить более подробный результат.

In [ ]:
total_trajectories = len(dataset.trajectory_lengths)

print("Total trajectories:", total_trajectories)

sampled_trajectories = np.random.choice(total_trajectories, 10)
print("Sampled trajectories:", sampled_trajectories)

all_mses = []

for traj_id in sampled_trajectories:
    mse = calc_mse_for_single_trajectory(
        pre_trained_policy,
        dataset,
        traj_id=traj_id,
        modality_keys=["right_arm", "right_hand"],   # we will only evaluate the right arm and right hand
        steps=150,
        action_horizon=16,
        plot=False
    )
    print(f"Trajectory {traj_id} MSE: {mse:.4f}")
    
    all_mses.append(mse)

print("====================================")
print("Mean MSE:", np.mean(all_mses))
print("Std MSE:", np.std(all_mses))


## Дообучение модели

Теперь дообучим модель на датасете. Не углубляясь в детали процесса, воспользуемся скриптом `gr00t_finetune.py`. Для дообучения можно выполнить следующую команду.

```bash
python scripts/gr00t_finetune.py --dataset-path ./demo_data/robot_sim.PickNPlace --num-gpus 1 --max-steps 500 --output-dir /tmp/gr00t-1/finetuned-model --data-config gr1_arms_only
```

_Полный список доступных параметров выводится командой `python scripts/gr00t_finetune.py --help`._

Скрипт сохранит дообученную модель в каталог `/tmp/gr00t-1/finetuned-model`. Мы загрузим дообученную модель с checkpoint на `500` шаге.

### Оценка дообученной модели

Теперь оценим дообученную модель, запустив на датасете политику и посмотрев, как она работает. Для оценки используем вспомогательную функцию. Это похоже на предыдущее руководство: [инференс предобученной модели](1-gr00t-inference.ipynb)

In [ ]:
finetuned_model_path = "/tmp/gr00t-1/finetuned-model/checkpoint-500"

from gr00t.utils.eval import calc_mse_for_single_trajectory
import warnings

finetuned_policy = Gr00tPolicy(
    model_path=finetuned_model_path,
    embodiment_tag="new_embodiment",
    modality_config=modality_config,
    modality_transform=modality_transform,
    device=device,
)

warnings.simplefilter("ignore", category=FutureWarning)

mse = calc_mse_for_single_trajectory(
    finetuned_policy,
    dataset,
    traj_id=0,
    modality_keys=["right_arm", "right_hand"],   # we will only evaluate the right arm and right hand
    steps=150,
    action_horizon=16,
    plot=True
)

print("MSE loss for trajectory 0:", mse)

Ура! Модель дообучена и оценена на датасете. Видно, что она освоила задачу и справляется с ней лучше предобученной модели.